# 🚀 KamiLimu 2026 Data Science Track
## Deploying ML Solutions & Safe Agentic AI Systems

Welcome to the practical lab on **Model Deployment & Agentic AI**! In this notebook, you will learn how to turn an offline machine learning model into a fully working, interactive application directly inside Google Colab:

1. **Packaging & Serializing ML Pipelines** using `scikit-learn` and `joblib`.
2. **Serving Models via REST APIs** using `FastAPI` and `Pydantic` with background execution.
3. **Building Safe Agentic AI Workflows** with **Guardrails**, **Tool Calling**, and **Grounded Knowledge Retrieval**.
4. **Launching an Interactive Streamlit UI** accessible directly from your browser via a public tunnel.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/ml_and_agentic_ai_deployment.ipynb)

Click **Open in Colab** to run this notebook in Google Colab. Then run the next cell so the `app/` files load.

--- 
## Step 1: Environment Setup in Colab
Run this cell to install all required libraries and tunneling tools.

In [ ]:
!pip install -q fastapi uvicorn pydantic streamlit scikit-learn pandas numpy joblib requests pyngrok nest-asyncio

--- 
## Step 2: Build & Serialize the Machine Learning Pipeline
We train a pipeline combining preprocessing (`ColumnTransformer`) and a `RandomForestRegressor`, then save the full artifact to `house_price_pipeline.joblib`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Create Ames Housing synthetic dataset
np.random.seed(42)
n = 1200
gr_liv_area = np.random.normal(1500, 500, n).clip(400, 4500)
overall_qual = np.random.randint(1, 11, n)
year_built = np.random.randint(1940, 2024, n)
total_bsmt_sf = np.random.normal(1000, 400, n).clip(0, 3000)
neighborhoods = np.random.choice(["NAmes", "CollgCr", "OldTown", "Edwards", "Somerst"], n)
house_styles = np.random.choice(["1Story", "2Story", "1.5Fin", "SLvl"], n)

price = (
    50000 
    + gr_liv_area * 75 
    + overall_qual * 15000 
    + (year_built - 1950) * 500 
    + total_bsmt_sf * 45 
    + np.random.normal(0, 15000, n)
)

df = pd.DataFrame({
    "GrLivArea": gr_liv_area,
    "OverallQual": overall_qual,
    "YearBuilt": year_built,
    "TotalBsmtSF": total_bsmt_sf,
    "Neighborhood": neighborhoods,
    "HouseStyle": house_styles,
    "SalePrice": price
})

# 2. Split Features & Target
numeric_features = ["GrLivArea", "OverallQual", "YearBuilt", "TotalBsmtSF"]
categorical_features = ["Neighborhood", "HouseStyle"]
target = "SalePrice"

X = df[numeric_features + categorical_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Pipeline
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42))
])

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)
print(f"Validation MAE: ${mean_absolute_error(y_test, preds):,.2f} | R²: {r2_score(y_test, preds):.4f}")

# 4. Save Artifact
joblib.dump(pipeline, "house_price_pipeline.joblib")
print("✅ Model Pipeline saved to 'house_price_pipeline.joblib'")

--- 
## Step 3: Write the FastAPI Service File
We write the `api.py` file to disk. It defines schema validation using `Pydantic` and provides a `/predict` endpoint.

In [ ]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import pandas as pd
import joblib

app = FastAPI(title="KamiLimu ML API", version="1.0.0")
model = joblib.load("house_price_pipeline.joblib")

class HouseFeatures(BaseModel):
    GrLivArea: float = Field(..., ge=300, le=8000, description="Living area sq ft")
    OverallQual: int = Field(..., ge=1, le=10, description="Quality 1-10")
    YearBuilt: int = Field(..., ge=1850, le=2026, description="Construction year")
    TotalBsmtSF: float = Field(..., ge=0, le=5000, description="Basement sq ft")
    Neighborhood: str = Field(default="NAmes")
    HouseStyle: str = Field(default="1Story")

class PredictionResponse(BaseModel):
    predicted_price: float
    currency: str = "USD"

@app.get("/")
def health():
    return {"status": "healthy", "service": "KamiLimu ML API"}

@app.post("/predict", response_model=PredictionResponse)
def predict(features: HouseFeatures):
    df = pd.DataFrame([features.model_dump()])
    price = model.predict(df)[0]
    return PredictionResponse(predicted_price=round(float(price), 2))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

--- 
## Step 4: Write the Agent Controller with Guardrails (`agent.py`)
This module implements the **ReAct Agent** with **Safety Guardrails** (preventing dosage/prescription hallucinations in health literacy) and **Tool Calling** (querying clinical guidelines or the ML prediction model).

In [ ]:
%%writefile agent.py
import re
import requests
from typing import Dict, Any, List, Optional

VERIFIED_HEALTH_KNOWLEDGE = {
    "invasive ductal carcinoma": {
        "definition": "Cancer that began in milk ducts and has grown into surrounding breast tissue.",
        "analogy": "Like plant roots extending beyond their initial container into nearby soil.",
        "question": "Ask your oncologist about tumour size, grade, and hormone receptor status."
    },
    "er-positive": {
        "definition": "Estrogen Receptor Positive (ER+). Tumour cells use estrogen to grow.",
        "analogy": "Estrogen acts like fuel for the cells; hormone therapies work by blocking this fuel.",
        "question": "Will hormone therapy (such as Tamoxifen) be part of my treatment plan?"
    },
    "her2-negative": {
        "definition": "HER2 Negative means the cells do not produce excess HER2 growth protein.",
        "analogy": "The growth signal is at a normal level rather than in overdrive.",
        "question": "What targeted therapies or chemotherapy regimens work best for HER2-negative tumours?"
    }
}

class SafetyGuardrails:
    PROHIBITED = [r"\b(dosage|dose|mg|prescribe|prescription|cure cancer)\b"]
    
    @classmethod
    def check_input(cls, text: str) -> tuple[bool, Optional[str]]:
        for pattern in cls.PROHIBITED:
            if re.search(pattern, text, re.IGNORECASE):
                return False, "⚠️ Guardrail Blocked: Prescription or dosage advice is strictly prohibited. Please consult your physician."
        return True, None

class KamiLimuAgent:
    def plan_and_execute(self, query: str) -> Dict[str, Any]:
        steps: List[str] = []
        is_safe, refusal = SafetyGuardrails.check_input(query)
        if not is_safe:
            return {"success": False, "steps": ["Input Guardrail: Blocked"], "response": refusal}
        
        steps.append("✅ Input passed safety checks.")
        
        # Check for ML pricing tool
        if any(w in query.lower() for w in ["house", "price", "valuation", "property", "sqft"]):
            steps.append("🧠 Intent: Property Valuation. Selecting `predict_house_price` tool.")
            match = re.search(r"(\d{3,5})\s*(?:sqft|sq ft)?", query, re.IGNORECASE)
            sqft = float(match.group(1)) if match else 1750.0
            
            try:
                res = requests.post("http://127.0.0.1:8000/predict", json={
                    "GrLivArea": sqft, "OverallQual": 7, "YearBuilt": 2012,
                    "TotalBsmtSF": 950.0, "Neighborhood": "NAmes", "HouseStyle": "1Story"
                }, timeout=3)
                if res.status_code == 200:
                    val = res.json()["predicted_price"]
                    response = f"Based on our ML model, estimated price for **{sqft:,.0f} sq ft** is **${val:,.2f} USD**."
                else:
                    response = f"ML Service Error: {res.text}"
            except Exception as e:
                response = f"Could not connect to FastAPI: {e}"
            return {"success": True, "steps": steps, "response": response}
        
        # Check for Medical Literacy Tool
        steps.append("🧠 Intent: Medical Comprehension. Querying verified knowledge base.")
        matches = [k for k in VERIFIED_HEALTH_KNOWLEDGE if k in query.lower()]
        if matches:
            body = "Here is a simplified explanation based on approved guidelines:\n\n"
            for k in matches:
                d = VERIFIED_HEALTH_KNOWLEDGE[k]
                body += f"### 📖 {k.title()}\n- **Definition:** {d['definition']}\n- **Analogy:** {d['analogy']}\n- **Question for Doctor:** *\"{d['question']}\"*\n\n"
            body += "---\n*🛡️ Educational summary only. Always consult your oncologist.*"
            return {"success": True, "steps": steps, "response": body}
        
        return {"success": True, "steps": steps, "response": "Ask about medical terms (e.g. 'ER-positive') or house prices (e.g. '2000 sqft house')."}

--- 
## Step 5: Write the Streamlit Web Application (`app.py`)
We create a multi-tab web application connecting directly to our FastAPI service and the AI Agent.

In [ ]:
%%writefile app.py
import streamlit as st
import requests
from agent import KamiLimuAgent

st.set_page_config(page_title="KamiLimu ML & AI Deployment", page_icon="🚀", layout="wide")
st.title("🚀 KamiLimu ML & Safe Agentic AI Portal")

if "agent" not in st.session_state:
    st.session_state.agent = KamiLimuAgent()

tab1, tab2 = st.tabs(["📊 Direct ML Prediction", "🤖 Agentic Assistant"])

with tab1:
    st.subheader("🏡 House Price Estimator")
    col1, col2 = st.columns(2)
    with col1:
        sqft = st.number_input("Living Area (sq ft)", min_value=300, max_value=8000, value=1750)
        qual = st.slider("Quality (1-10)", 1, 10, 7)
        year = st.number_input("Year Built", 1880, 2026, 2012)
    with col2:
        bsmt = st.number_input("Basement Area (sq ft)", 0, 5000, 950)
        neigh = st.selectbox("Neighborhood", ["NAmes", "CollgCr", "OldTown", "Edwards", "Somerst"])
        style = st.selectbox("House Style", ["1Story", "2Story", "1.5Fin", "SLvl"])
    
    if st.button("⚡ Get Valuation", type="primary"):
        payload = {"GrLivArea": sqft, "OverallQual": qual, "YearBuilt": year, "TotalBsmtSF": bsmt, "Neighborhood": neigh, "HouseStyle": style}
        try:
            res = requests.post("http://127.0.0.1:8000/predict", json=payload, timeout=5)
            if res.status_code == 200:
                st.success(f"### 🏷️ Estimated Value: **${res.json()['predicted_price']:,.2f} USD**")
            else:
                st.error(f"Error: {res.text}")
        except Exception as e:
            st.error(f"Could not reach FastAPI: {e}")

with tab2:
    st.subheader("🤖 Safe Medical & Analytics Agent")
    if "messages" not in st.session_state:
        st.session_state.messages = []
    
    for m in st.session_state.messages:
        with st.chat_message(m["role"]):
            st.markdown(m["content"])
    
    prompt = st.chat_input("Ask a question (e.g. 'What is ER-positive?' or 'Valuation for 2200 sqft house')...")
    if prompt:
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)
        
        with st.chat_message("assistant"):
            out = st.session_state.agent.plan_and_execute(prompt)
            with st.expander("🧠 Agent Thought Trace"):
                for s in out.get("steps", []):
                    st.write(s)
            st.markdown(out["response"])
            st.session_state.messages.append({"role": "assistant", "content": out["response"]})

--- 
## Step 6: Start FastAPI & Launch Streamlit in Google Colab

### 6a — Set your ngrok token (run this first)
1. Get a free token at [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
2. In Colab, click the **🔑 Secrets** icon in the left sidebar → **Add new secret** → name: `NGROK_TOKEN`, value: your token → enable notebook access.
3. Run the cell below — it loads the secret into `os.environ` for the launch cell.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["NGROK_TOKEN"] = userdata.get("NGROK_TOKEN")
    print("✅ NGROK_TOKEN loaded from Colab Secrets.")
except Exception:
    # Outside Colab — paste your token directly here (do not commit this file after):
    # os.environ["NGROK_TOKEN"] = "paste_your_token_here"
    print("⚠️  Not running in Colab. Uncomment the line above and paste your token.")

### 6b — Launch servers and open tunnel

In [ ]:
import subprocess
import time
import socket
from pyngrok import ngrok
import os

def free_port(port, timeout=10):
    """Kill any process on `port` and wait until the port is actually free."""
    subprocess.run(
        f"lsof -ti tcp:{port} | xargs -r kill -9 2>/dev/null || true",
        shell=True
    )
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                pass
            time.sleep(0.3)  # port still occupied, keep waiting
        except OSError:
            return  # port is free
    raise RuntimeError(f"Port {port} did not free up within {timeout}s")

# 0. Release ports and any stale ngrok tunnels
ngrok.kill()
free_port(8000)
free_port(8501)

# 1. Start FastAPI in background
subprocess.Popen(["uvicorn", "api:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(2)
print("🟢 FastAPI Server running in background on port 8000.")

# 2. Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)
print("🟢 Streamlit Server running in background on port 8501.")

# 3. Open a public tunnel to Streamlit via ngrok
token = os.getenv("NGROK_TOKEN") or userdata.get('NGROK_TOKEN')
if not token:
    raise ValueError("NGROK_TOKEN is not set in the environment variables. Please set it before running.")

ngrok.set_auth_token(token)
public_url = ngrok.connect(8501)
print(f"\n🌐 Open your live app here → {public_url}")